# Evaluación de Modelos


## 1. Setup

In [ ]:
# 1. Clonar
!git clone https://github.com/igcondor/SCY1101_Analisis_Semestral_2026.git

# 2. Entrar a la carpeta
%cd /content/SCY1101_Analisis_Semestral_2026

# 3. Cambiarse al branch correcto ANTES de cualquier import
!git checkout condore_entrena_super

# 4. Traer cambios más recientes
!git pull

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import (accuracy_score, classification_report,
                             mean_squared_error, r2_score,
                             confusion_matrix, ConfusionMatrixDisplay,
                             precision_score, recall_score, f1_score)

import sys
sys.path.append('.')
from src.preprocesamiento_data import preprocesar, pipeline_clasificacion, pipeline_regresion
from src.model_training import entrenar_clasificacion, entrenar_regresion
from src.model_evaluation import (evaluar_clasificacion_cv, evaluar_regresion_cv,
                                   tabla_comparacion_clf, tabla_comparacion_reg)
SEED = 42

## Preparación de Datos

In [ ]:
df = preprocesar('retail_store_sales.csv')

X_clf_desc, y_clf_desc = pipeline_clasificacion(df)
X_train_clf_desc, X_test_clf_desc, y_train_clf_desc, y_test_clf_desc = train_test_split(
    X_clf_desc, y_clf_desc, test_size=0.2, random_state=SEED)

X_reg_ts, y_reg_ts = pipeline_regresion(df)
X_train_reg_ts, X_test_reg_ts, y_train_reg_ts, y_test_reg_ts = train_test_split(
    X_reg_ts, y_reg_ts, test_size=0.2, random_state=SEED)

## Reentrenamiento de Modelos

Reentrenamos los mismos modelos del notebook 02 con los mismos parámetros y SEED para garantizar reproducibilidad.

In [ ]:
# Clasificación
arbol_clf = DecisionTreeClassifier(random_state=SEED)
arbol_clf, y_pred_arbol = entrenar_clasificacion(
    arbol_clf, X_train_clf_desc, X_test_clf_desc,
    y_train_clf_desc, y_test_clf_desc, "Árbol de Decisión")

rf_clf = RandomForestClassifier(n_estimators=100, random_state=SEED)
rf_clf, y_pred_rf_clf = entrenar_clasificacion(
    rf_clf, X_train_clf_desc, X_test_clf_desc,
    y_train_clf_desc, y_test_clf_desc, "Random Forest")

lr_clf = LogisticRegression(random_state=SEED, max_iter=1000)
lr_clf, y_pred_lr_clf = entrenar_clasificacion(
    lr_clf, X_train_clf_desc, X_test_clf_desc,
    y_train_clf_desc, y_test_clf_desc, "Regresión Logística")

# Regresión
lr_reg = LinearRegression()
lr_reg, y_pred_lr_reg = entrenar_regresion(
    lr_reg, X_train_reg_ts, X_test_reg_ts,
    y_train_reg_ts, y_test_reg_ts, "Regresión Lineal")

rf_reg = RandomForestRegressor(n_estimators=100, random_state=SEED)
rf_reg, y_pred_rf_reg = entrenar_regresion(
    rf_reg, X_train_reg_ts, X_test_reg_ts,
    y_train_reg_ts, y_test_reg_ts, "Random Forest Regresión")

## Validación Cruzada clasificacion


In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

modelos_clf = {
    'Árbol de Decisión': DecisionTreeClassifier(random_state=SEED),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=SEED),
    'Regresión Logística': LogisticRegression(random_state=SEED, max_iter=1000)
}

resultados_cv_clf = evaluar_clasificacion_cv(modelos_clf, X_clf_desc, y_clf_desc, kf)
print(resultados_cv_clf.to_string(index=False))

## Metricas Detalladas clasificación

Además del accuracy, evaluamos **precision**, **recall** y **F1-score**:
- **Precision:** de los que predijo como True, cuántos eran realmente True
- **Recall:** de los que eran realmente True, cuántos detectó correctamente
- **F1-score:** promedio armónico entre precision y recall

In [ ]:
resultados_metricas_clf = tabla_comparacion_clf(
    modelos_clf={
        'Árbol de Decisión': (arbol_clf, y_pred_arbol),
        'Random Forest': (rf_clf, y_pred_rf_clf),
        'Regresión Logística': (lr_clf, y_pred_lr_clf)
    },
    y_test=y_test_clf_desc
)
print(resultados_metricas_clf.to_string(index=False))

## Matrices de Confusión  clasificación

La matriz de confusión muestra cuántas predicciones fueron correctas e incorrectas por clase.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (nombre, y_pred) in zip(axes, [
    ('Árbol de Decisión', y_pred_arbol),
    ('Random Forest', y_pred_rf_clf),
    ('Regresión Logística', y_pred_lr_clf)
]):
    cm = confusion_matrix(y_test_clf_desc, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Sin Descuento', 'Con Descuento'])
    disp.plot(ax=ax, colorbar=False)
    ax.set_title(nombre)

plt.tight_layout()
plt.show()

## Validación Cruzada — Regresión

In [ ]:
modelos_reg = {
    'Regresión Lineal': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=SEED)
}

resultados_cv_reg = evaluar_regresion_cv(modelos_reg, X_reg_ts, y_reg_ts, kf)
print(resultados_cv_reg.to_string(index=False))

## 8. Métricas Detalladas  Regresion

- **RMSE:** error promedio en las mismas unidades del target (Total Spent). Más bajo es mejor.
- **R2:** proporción de varianza explicada por el modelo. Más cercano a 1 es mejor.

In [ ]:
resultados_metricas_reg = tabla_comparacion_reg(
    modelos_reg={
        'Regresión Lineal': (lr_reg, y_pred_lr_reg),
        'Random Forest': (rf_reg, y_pred_rf_reg)
    },
    y_test=y_test_reg_ts
)
print(resultados_metricas_reg.to_string(index=False))

## 9. Comparación Visual

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Clasificación — Accuracy por modelo
nombres_clf = resultados_cv_clf['Modelo'].tolist()
accuracy_clf = resultados_cv_clf['Accuracy Media'].tolist()
axes[0].barh(nombres_clf, accuracy_clf, color='steelblue')
axes[0].set_title('Accuracy Promedio (CV) — Clasificación')
axes[0].set_xlabel('Accuracy')
axes[0].set_xlim(0, 1)
for i, v in enumerate(accuracy_clf):
    axes[0].text(v + 0.01, i, f'{v:.4f}', va='center')

# Regresión — R2 por modelo
nombres_reg = resultados_cv_reg['Modelo'].tolist()
r2_reg = resultados_cv_reg['R2 Medio'].tolist()
axes[1].barh(nombres_reg, r2_reg, color='seagreen')
axes[1].set_title('R2 Promedio (CV) — Regresión')
axes[1].set_xlabel('R2')
axes[1].set_xlim(0, 1)
for i, v in enumerate(r2_reg):
    axes[1].text(v + 0.01, i, f'{v:.4f}', va='center')

plt.tight_layout()
plt.show()

## 10. Conclusiones

### Clasificación
Los 3 modelos obtuvieron accuracy cercano al 50%, lo que indica que `Discount Applied` 
no tiene una relación predecible con las variables disponibles en el dataset. 
Este es un hallazgo válido del análisis — no todos los problemas son predecibles 
con los datos disponibles.

### Regresión
Random Forest superó a Regresión Lineal en R2 y RMSE, confirmando que existen 
relaciones no lineales entre los features y `Total Spent`. 
Ambos modelos muestran un rendimiento aceptable para el problema.

El análisis de hiperparámetros se realiza en el notebook 04.